# Thermal Stack

Le pipeline est dans `thermal_stack.py`. Les donnees marche (spot, consommation 30 min, fuel cocktail) sont chargees via `data_loader.py`.

In [1]:
import pandas as pd
from IPython.display import display

from thermal_stack import (
    ThermalStackConfig,
    MarginalCostConfig,
    StackAnalysisConfig,
    build_thermal_stack,
    merit_order_for_areas,
    plot_merit_order_plotly,
    plot_regional_merit_order_plotly,
    identify_price_setters,
    summarize_price_setters,
    plot_price_setter_share_plotly,
    PriceSetterConfig,
    default_price_setter_configs,
    default_marginal_cost_configs,
    default_parallel_stack_configs,
    run_parallel_stack_analysis,
    summarize_parallel_stack_analysis,
    compare_price_setter_demand_levels,
    plot_price_setter_config_comparison,
    COST_METHOD_OPERATIONAL_JPY,
    COST_METHOD_ARTICLE_JPY,
    LNG_PRICE_JLC,
    KANSAI_FENCE_AREAS,
)
import data_loader

In [2]:
operational_cost = MarginalCostConfig(
    "operational_jlc_jpy",
    COST_METHOD_OPERATIONAL_JPY,
    LNG_PRICE_JLC,
)
article_cost = MarginalCostConfig(
    "article_jpy",
    COST_METHOD_ARTICLE_JPY,
    LNG_PRICE_JLC,
    zero_mc_fuels=("Hydro",),
)
article_vre_cost = MarginalCostConfig(
    "article_jpy_vre",
    COST_METHOD_ARTICLE_JPY,
    LNG_PRICE_JLC,
    zero_mc_fuels=("Hydro", "Solar", "Wind"),
)

result = build_thermal_stack(
    ThermalStackConfig(
        delivery_month="2026-04",
        export_excel=False,
        load_unit_production=False,
        marginal_cost_config=operational_cost,
    )
)
print(f"Delivery month: {result.delivery_month:%Y-%m}")
print(f"Thermal stack rows: {len(result.merit_order):,}")
display(result.fuel_cocktail[[c for c in result.fuel_cocktail.columns if 'jpy' in c.lower()]].tail(3))

c:\Develop\data_loader.py:375: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
c:\Develop\data_loader.py:375: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors="coerce")


Delivery month: 2026-04
Thermal stack rows: 517


,EURJPY,coal_jpn_cif_jpy_t,oil_jpn_cif_jpy_kl,jcc_jpykwh,jlc_jpykwh,jkm_jpykwh,ttf_jpykwh,coal_cif_jpykwh
date,,,,,,,,
2026-06-24,183.35,19010.666455,13855.871151,1.354620,1.235882,0.735462,0.657686,0.253249
2026-06-25,183.35,19049.298107,13933.666457,1.362225,1.242084,0.735462,0.639575,0.253763
2026-06-26,183.35,19087.598220,14009.237695,1.369614,1.248061,0.735462,0.639575,0.254274


## Merit Order (JPY/kWh)

In [3]:

region = "Kansai"
mo_region = merit_order_for_areas(result, areas=region, cost_config=operational_cost)
median_snapshot = identify_price_setters(mo_region, config=PriceSetterConfig(
    "Kansai | consumption", ("Kansai",), "Kansai", "consumption"
), start=result.delivery_month, end=result.delivery_month + pd.offsets.MonthEnd(0))

fig = plot_merit_order_plotly(
    mo_region,
    region=region,
    period=result.delivery_month,
    demand_gw=median_snapshot["demand_gw"].median(),
    spot_price_jpy_kwh=median_snapshot["spot_jpy_kwh"].median(),
    title=f"Merit Order ({region}) - JPY/kWh - {result.delivery_month:%B %Y}",
)
fig.show()

In [4]:
fig_regional = plot_regional_merit_order_plotly(result)
fig_regional.show()

# Modes paralleles
Combinaison de:

Cout marginal: operational JLC / article (alpha + C_f) / article + VRE (hydro, solar, wind a MC=0)
Region + demande: Kansai vs fence Kansai+Hokuriku+Chubu x consumption / residual_load / thermal_residual
Prix spot et stack en JPY/kWh.

In [5]:
parallel_cost_modes = [operational_cost, article_cost, article_vre_cost]
parallel_region_modes = default_price_setter_configs(include_kansai_fence=True)
parallel_configs = default_parallel_stack_configs(
    include_kansai_fence=True,
    cost_configs=parallel_cost_modes,
    price_setter_configs=parallel_region_modes,
)

parallel_results = run_parallel_stack_analysis(
    result,
    analysis_configs=parallel_configs,
    start=result.delivery_month,
    end=result.delivery_month + pd.offsets.MonthEnd(0),
)
parallel_summary = summarize_parallel_stack_analysis(parallel_results)
display(parallel_summary.round(3))

,share,method,fuel_class,analysis,cost_mode,cost_methodology,lng_price_source,demand_basis,areas
0,0.668,demand,LNG,operational_jlc_jpy :: Kansai | consumption,operational_jlc_jpy,operational_jpy,jlc,consumption,Kansai
1,0.332,demand,Nuclear,operational_jlc_jpy :: Kansai | consumption,operational_jlc_jpy,operational_jpy,jlc,consumption,Kansai
2,0.928,price,Oil,operational_jlc_jpy :: Kansai | consumption,operational_jlc_jpy,operational_jpy,jlc,consumption,Kansai
3,0.052,price,Coal,operational_jlc_jpy :: Kansai | consumption,operational_jlc_jpy,operational_jpy,jlc,consumption,Kansai
4,0.017,price,LNG,operational_jlc_jpy :: Kansai | consumption,operational_jlc_jpy,operational_jpy,jlc,consumption,Kansai
...,...,...,...,...,...,...,...,...,...
112,0.006,demand,Wind,article_jpy_vre :: Kansai+Hokuriku+Chubu | the...,article_jpy_vre,article_jpy,jlc,thermal_residual,"Hokuriku, Kansai, Chubu"
113,0.916,price,Oil,article_jpy_vre :: Kansai+Hokuriku+Chubu | the...,article_jpy_vre,article_jpy,jlc,thermal_residual,"Hokuriku, Kansai, Chubu"
114,0.050,price,Solar,article_jpy_vre :: Kansai+Hokuriku+Chubu | the...,article_jpy_vre,article_jpy,jlc,thermal_residual,"Hokuriku, Kansai, Chubu"
115,0.032,price,LNG,article_jpy_vre :: Kansai+Hokuriku+Chubu | the...,article_jpy_vre,article_jpy,jlc,thermal_residual,"Hokuriku, Kansai, Chubu"


In [6]:

# Comparer operational vs article sur Kansai seul | thermal_residual
focus = parallel_summary[
    (parallel_summary["areas"] == "Kansai")
    & parallel_summary["analysis"].str.contains("thermal_residual", na=False)
]
if focus.empty:
    raise ValueError("Aucune ligne pour Kansai | thermal_residual dans parallel_summary.")
fig_focus = plot_price_setter_config_comparison(
    focus,
    title=f"Kansai thermal residual - cost modes - {result.delivery_month:%B %Y}",
)
fig_focus.show()

KeyError: 'config'

## Marginal price setter - configurations paralleles

Comparaison de 6 configurations:
- **Region**: Kansai seul vs fence **Kansai + Hokuriku + Chubu**
- **Demande**:
  - `consumption` = consommation brute
  - `residual_load` = consommation - solaire - eolien
  - `thermal_residual` = consommation - solaire - eolien - inter_flows - hydro - nucleaire

Le prix spot de reference reste **Kansai** (`jpKan`) pour toutes les configs fence.

In [ ]:
configs = default_price_setter_configs(include_kansai_fence=True)
config_results = run_price_setter_configs(
    result,
    configs=configs,
    start=result.delivery_month,
    end=result.delivery_month + pd.offsets.MonthEnd(0),
)

demand_comparison = compare_price_setter_demand_levels(config_results)
summary_comparison = summarize_price_setter_configs(config_results)

display(demand_comparison.round(2))
display(summary_comparison.round(3))

,config,areas,demand_basis,demand_gw_min,demand_gw_median,demand_gw_max,spot_eur_mwh_median
0,Kansai | consumption,Kansai,consumption,10.09,13.09,17.36,85.72
1,Kansai | residual_load,Kansai,residual_load,6.68,12.16,16.50,85.72
2,Kansai | thermal_residual,Kansai,thermal_residual,-1.42,5.03,8.43,85.72
3,Kansai+Hokuriku+Chubu | consumption,"Hokuriku, Kansai, Chubu",consumption,21.34,28.24,37.60,85.72
4,Kansai+Hokuriku+Chubu | residual_load,"Hokuriku, Kansai, Chubu",residual_load,11.17,26.00,35.68,85.72
5,Kansai+Hokuriku+Chubu | thermal_residual,"Hokuriku, Kansai, Chubu",thermal_residual,-0.02,14.39,22.76,85.72


,share,method,fuel_class,config,demand_basis,areas
0,0.882,demand,LNG,Kansai | consumption,consumption,Kansai
1,0.118,demand,Coal,Kansai | consumption,consumption,Kansai
2,0.574,price,NaN,Kansai | consumption,consumption,Kansai
3,0.309,price,LNG,Kansai | consumption,consumption,Kansai
4,0.069,price,Coal,Kansai | consumption,consumption,Kansai
5,0.030,price,Nuclear,Kansai | consumption,consumption,Kansai
6,0.019,price,Oil,Kansai | consumption,consumption,Kansai
7,0.679,demand,LNG,Kansai | residual_load,residual_load,Kansai
8,0.321,demand,Coal,Kansai | residual_load,residual_load,Kansai
9,0.574,price,NaN,Kansai | residual_load,residual_load,Kansai


In [ ]:
fig_compare = plot_price_setter_config_comparison(
    summary_comparison,
    title=f"Marginal fuel share by configuration - {result.delivery_month:%B %Y}",
)
fig_compare.show()

In [ ]:
# Focus Kansai: impact du fence et de la base de demande
kansai_configs = [cfg for cfg in configs if "Kansai" in cfg.name]
for cfg in kansai_configs:
    summary = summarize_price_setters(config_results[cfg.name])
    fig = plot_price_setter_share_plotly(
        summary,
        region=cfg.name,
        title=f"{cfg.name} - {result.delivery_month:%B %Y}",
    )
    fig.show()

In [ ]:
# Merit order fence Kansai+Hokuriku+Chubu avec thermal residual median
fence_cfg = next(cfg for cfg in configs if cfg.name == "Kansai+Hokuriku+Chubu | thermal_residual")
mo_fence = merit_order_for_areas(result, areas=list(fence_cfg.areas))
fence_setters = config_results[fence_cfg.name]
median_demand = fence_setters["demand_gw"].median()
median_spot = fence_setters["spot_eur_mwh"].median()

fig_fence = plot_merit_order_plotly(
    mo_fence,
    region=list(fence_cfg.areas),
    period=result.delivery_month,
    demand_gw=median_demand,
    spot_price_eur_mwh=median_spot,
    title=f"Merit Order ({', '.join(fence_cfg.areas)}) - thermal residual - {result.delivery_month:%B %Y}",
)
fig_fence.show()